In [1]:
import pandas as pd

In [2]:
#load data 

df_smi = pd.read_csv("DataSets/SSMI.csv", parse_dates=["Date"], index_col= "Date")

df_smi

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1990-11-09,1378.900024,1389.000000,1375.300049,1387.099976,1387.099976,0.0
1990-11-12,1388.099976,1408.099976,1388.099976,1407.500000,1407.500000,0.0
1990-11-13,1412.199951,1429.400024,1411.400024,1415.199951,1415.199951,0.0
1990-11-14,1413.599976,1413.599976,1402.099976,1410.300049,1410.300049,0.0
1990-11-15,1410.599976,1416.699951,1405.099976,1405.699951,1405.699951,0.0
...,...,...,...,...,...,...
2021-05-10,11157.759766,11159.080078,11086.980469,11123.669922,11123.669922,40932200.0
2021-05-11,NaN,NaN,NaN,NaN,NaN,NaN
2021-05-12,11014.330078,11069.240234,10981.799805,11033.900391,11033.900391,47616100.0


In [3]:
#print missing values

df_smi.isna()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1990-11-09,False,False,False,False,False,False
1990-11-12,False,False,False,False,False,False
1990-11-13,False,False,False,False,False,False
1990-11-14,False,False,False,False,False,False
1990-11-15,False,False,False,False,False,False
...,...,...,...,...,...,...
2021-05-10,False,False,False,False,False,False
2021-05-11,True,True,True,True,True,True
2021-05-12,False,False,False,False,False,False


In [4]:
#count missing values

df_smi.isna().sum()

Open         157
High         157
Low          157
Close        157
Adj Close    157
Volume       157
dtype: int64

In [5]:
# show the rows where date is fixed Swiss holidays and isna()
holiday_mask = df_smi.isna().any(axis=1) & (
    ((df_smi.index.month == 1) & (df_smi.index.day.isin([1, 2]))) |   # Neujahr, Berchtoldstag
    ((df_smi.index.month == 5) & (df_smi.index.day == 1)) |           # Tag der Arbeit
    ((df_smi.index.month == 8) & (df_smi.index.day == 1)) |           # Bundesfeiertag
    ((df_smi.index.month == 12) & (df_smi.index.day.isin([24, 25, 26, 31])))  # Heiligabend, Weihnachten, Stephanstag, Silvester
)
df_smi[holiday_mask]

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1990-12-25,NaN,NaN,NaN,NaN,NaN,NaN
1990-12-26,NaN,NaN,NaN,NaN,NaN,NaN
1990-12-31,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-01,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-02,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2006-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2006-05-01,NaN,NaN,NaN,NaN,NaN,NaN
2006-08-01,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Bewegliche Feiertage (Oster-basiert) identifizieren
from dateutil.easter import easter

moving_holidays = set()
for year in df_smi.index.year.unique():
    e = pd.Timestamp(easter(year))
    moving_holidays.add(e - pd.Timedelta(days=2))   # Karfreitag
    moving_holidays.add(e + pd.Timedelta(days=1))   # Ostermontag
    moving_holidays.add(e + pd.Timedelta(days=39))  # Auffahrt
    moving_holidays.add(e + pd.Timedelta(days=50))  # Pfingstmontag

moving_mask = df_smi.index.isin(moving_holidays) & df_smi.isna().any(axis=1)
df_smi[moving_mask]

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1991-03-29,NaN,NaN,NaN,NaN,NaN,NaN
1991-04-01,NaN,NaN,NaN,NaN,NaN,NaN
1991-05-09,NaN,NaN,NaN,NaN,NaN,NaN
1991-05-20,NaN,NaN,NaN,NaN,NaN,NaN
1992-04-17,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2005-05-16,NaN,NaN,NaN,NaN,NaN,NaN
2006-04-14,NaN,NaN,NaN,NaN,NaN,NaN
2006-04-17,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Alle Feiertage (fix + beweglich) aus dem DataFrame entfernen
df_smi = df_smi.drop(df_smi[holiday_mask | moving_mask].index)
print(f"Shape nach Entfernung: {df_smi.shape}")
print(f"Verbleibende NaN-Zeilen: {df_smi.isna().any(axis=1).sum()}")

Shape nach Entfernung: (7678, 6)
Verbleibende NaN-Zeilen: 14


In [8]:
#show NAN values

df_smi[df_smi.isnull().any(axis=1)]

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1992-09-14,NaN,NaN,NaN,NaN,NaN,NaN
1992-12-04,NaN,NaN,NaN,NaN,NaN,NaN
1994-12-30,NaN,NaN,NaN,NaN,NaN,NaN
1995-12-29,NaN,NaN,NaN,NaN,NaN,NaN
1999-11-12,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN
2001-09-11,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2021-01-22,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# forward fill missing values open, high, low, close, adj close, volume
df_smi[["Open", "High", "Low", "Close", "Adj Close", "Volume"]] = df_smi[["Open", "High", "Low", "Close", "Adj Close", "Volume"]].ffill()

In [10]:
#show NAN values

df_smi[df_smi.isnull().any(axis=1)]

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,


In [11]:
# safe df_smi as new CSV
df_smi.to_csv("DataSets/SSMI_cleaned.csv")
print("DataFrame saved successfully to DataSets/SSMI_cleaned.csv")

DataFrame saved successfully to DataSets/SSMI_cleaned.csv
